In [0]:
%sql
show catalogs;
use catalog databricks_project;
show schemas;
use schema schema;

In [0]:
from pyspark.sql.functions import initcap, col, lower
df_bronze = spark.table("bronze_products")
silver_products_df = df_bronze.withColumn("category" , initcap(lower(col("category"))))
(
    silver_products_df.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("databricks_project.schema.silver_products")
)

In [0]:
from pyspark.sql.functions import regexp_replace, col,to_timestamp, to_utc_timestamp, trim

df_bronze = spark.table("bronze_transactions").where(col("corrupted_flag") != "Y")

df_silver_transactions = (
    df_bronze
    .withColumn("price", regexp_replace(col("price"), "[$, ]", "").cast("decimal(10,2)"))
    .withColumn("order_timestamp_utc", to_utc_timestamp(to_timestamp(col("order_timestamp"), "yyyy-MM-dd HH:mm:ss z"), "PST"))
)

(
    df_silver_transactions
    .write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable("databricks_project.schema.silver_transactions"))

In [0]:
%sql
select * from databricks_project.schema.silver_transactions;

select * from databricks_project.schema.silver_products

item_id,product_name,category,_ingest_timestamp,_source_file_name
PROD-001,Laptop,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-002,Smartphone,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-003,Headphones,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-004,Tablet,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-005,Keyboard,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-006,Monitor,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-007,Mouse,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-008,Webcam,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-009,Speaker,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
PROD-010,Printer,Electronics,2026-04-09T07:25:41.749Z,dbfs:/Volumes/databricks_project/schema/raw_data_volume/products_raw.csv
